# 🔧 Custom Tools - Strands Agents

Learn how to create and use custom tools to extend your agent's capabilities.

## What are Tools?

Tools are functions that agents can call to perform specific tasks:
- 📊 Data processing
- 🌐 API calls
- 📁 File operations
- 🧮 Calculations
- And more!

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
from datetime import datetime

print("✅ Imports successful!")

## Using Tools with Agents

Let's create a calculator tool with the `@tool` decorator and give it to an agent.

> Note: earlier versions of this notebook imported `calculator` from `strands_tools`. That tool was removed in `strands-agents-tools` 0.9.0 (see the [community tools migration guide](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/community-tools-package/)), so the calculator is now a small custom tool defined here.


In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluates a basic arithmetic expression.

    Args:
        expression: Arithmetic expression using numbers, + - * / ( ) and the ** operator,
            for example "156 * 23" or "(50 + 75) / 5".

    Returns:
        The numeric result as a string, or an error message.
    """
    import ast, operator

    ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ops:
            return ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in ops:
            return ops[type(node.op)](_eval(node.operand))
        raise ValueError("Unsupported expression")

    try:
        return str(_eval(ast.parse(expression, mode="eval").body))
    except Exception as e:
        return f"Error: {e}"


# Setup Bedrock model (Claude Haiku 5.5, cross-region inference profile)
session = boto3.Session(region_name='us-east-1')
bedrock_model = BedrockModel(
    model_id="global.anthropic.claude-haiku-5-5",
    boto_session=session
)

# Create agent with calculator tool
math_agent = Agent(
    model=bedrock_model,
    tools=[calculator],
    system_prompt="You are a helpful math assistant. Use the calculator tool to perform calculations."
)

print("✅ Math agent created with calculator tool!")


In [ ]:
# Test the calculator tool
response = math_agent("What is 156 multiplied by 23?")
print(response)

In [ ]:
# Inspect the AgentResult object
print(f"Message: {response.message}")
print("-" * 100 + "\n")

print(f"Metrics: {response.metrics}")
print("-" * 100 + "\n")

print(f"State: {response.state}")
print("-" * 100 + "\n")

print(f"Stop Reason: {response.stop_reason}")
print("-" * 100 + "\n")

## Creating More Complex Tools

Let's create a tool that gets current time information:

In [ ]:
@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Gets the current date and time.
    
    Args:
        timezone: The timezone (currently only UTC supported)
    
    Returns:
        Current date and time as a string
    """
    now = datetime.now()
    return f"Current time ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"

print("✅ Time tool created!")

In [ ]:
# Create agent with multiple tools
assistant = Agent(
    model=bedrock_model,
    tools=[calculator, get_current_time],
    system_prompt="You are a helpful assistant with access to calculator and time tools."
)

response = assistant("What time is it? Also, what's 50 plus 75?")
print(response)

## Using Built-in Tools

The Strands Agents SDK ships a set of *vended tools* in `strands.vended_tools` ([docs](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/community-tools-package/)). Here we use:

- `read`: reads a file by absolute path. Text files come back as numbered lines; images (png/jpg/gif/webp) and documents (pdf/docx/xlsx...) come back as media blocks the model can look at directly.
- `video_reader_local`: a custom tool from this repo that sends a video to Amazon Nova 2 Lite through the Bedrock Converse API ([Nova video understanding](https://docs.aws.amazon.com/nova/latest/userguide/modalities-video.html)).

> `image_reader` and `file_read` from `strands_tools` were removed in 0.9.0; `read` replaces both.


In [ ]:
import os
from strands.vended_tools import read
from multimodal_understanding.video_reader_local import video_reader_local

# Vended `read` needs absolute paths
DATA = os.path.abspath("multimodal_understanding/data-sample")

# Create agent with built-in tools
multimodal_agent = Agent(
    model=bedrock_model,
    tools=[read, video_reader_local],
    system_prompt=f"""You are a multi-modal assistant that can:
    - Read and analyze images and documents (PDF, CSV, DOCX, etc.) with the `read` tool
    - Analyze videos with the `video_reader_local` tool
    - Use advanced reasoning for complex tasks.
    Files live in {DATA}; always pass absolute paths to the tools.
    """
)

print("✅ Multi-modal agent created with built-in tools!")


In [ ]:
# Example 1: Image analysis
print("=== 📸 IMAGE ANALYSIS ===")
image_result = multimodal_agent(f"Analyze the image {DATA}/diagram.jpg in detail and describe everything you observe")
print(image_result)
print("\n" + "="*80 + "\n")


In [ ]:
# Example 2: Document analysis (if you have a PDF document)
print("=== 📄 DOCUMENT ANALYSIS ===")
doc_result = multimodal_agent(f"Summarize as json the content of the document {DATA}/Welcome-Strands-Agents-SDK.pdf")
print(doc_result)


In [ ]:
# Example 3: Video analysis
print("=== 🎬 VIDEO ANALYSIS ===")
video_result = multimodal_agent(f"Analyze the video {DATA}/moderation-video.mp4 and describe in detail the actions and scenes you observe")
print(video_result)
print("\n" + "="*80 + "\n")


In [ ]:
# Inspect the AgentResult object
print(f"Message: {video_result.message}")
print("-" * 100 + "\n")

print(f"Metrics: {video_result.metrics}")
print("-" * 100 + "\n")

print(f"State: {video_result.state}")
print("-" * 100 + "\n")

print(f"Stop Reason: {video_result.stop_reason}")
print("-" * 100 + "\n")

## Direct Tool Usage

You can also call tools directly without the agent:

In [ ]:
# Example 4. Direct use of tools
video_analysis = multimodal_agent.tool.video_reader_local(
     video_path=f"{DATA}/moderation-video.mp4",
     text_prompt="What are the main elements in this video?"
)


In [ ]:
video_analysis

In [ ]:
# Same agent, importing the tool from the multimodal_understanding package path
from multimodal_understanding.video_reader_local import video_reader_local

multimodal_agent = Agent(
    model=bedrock_model,
    tools=[read, video_reader_local],
    system_prompt=f"""You are a multi-modal assistant that can:
    - Read and analyze images and documents with the `read` tool
    - Analyze videos with the `video_reader_local` tool
    Files live in {DATA}; always pass absolute paths to the tools.
    """
)

print("✅ Multi-modal agent created with built-in tools!")


In [ ]:
# Example 5. Direct use of tools
video_analysis = multimodal_agent.tool.video_reader_local(
     video_path=f"{DATA}/moderation-video.mp4",
     text_prompt="What are the main elements in this video?"
)


In [ ]:
video_analysis

## Summary

In this notebook, you learned:

✅ How to create custom tools with the `@tool` decorator

✅ How to add tools to agents

✅ How to use the SDK vended tools (`strands.vended_tools.read`)

✅ How to create agents with multiple tools

✅ How to call tools directly


### Next Steps

Continue to the next notebook to learn about Model Context Protocol (MCP) integration!